# 10.6 Python

Chapter 10 develops the law of large numbers, the central limit theorem, and related inequalities. This section uses Python to verify Jensen's inequality numerically, visualize the LLN as a running average, estimate $\pi$ by Monte Carlo integration, illustrate the CLT through histograms of sample means, and work with the Chi-Square and Student-$t$ distributions.

In [ ]:
import numpy as np
import scipy.stats as st
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(palette="deep")
rng = np.random.default_rng(seed=42)

## Jensen's Inequality

Jensen's inequality states that for a convex function $g$ and a random variable $X$,

$$E[g(X)] \geq g(E[X]),$$

with the inequality reversed when $g$ is concave. Simulation lets us check specific instances. For $X \sim \text{Expo}(1)$, the function $g(x) = \log x$ is concave, so we expect $E(\log X) \leq \log E(X)$. The true values are $E(\log X) = -\gamma \approx -0.5772$ (the negative of the Euler–Mascheroni constant) and $\log E(X) = \log 1 = 0$.

In [ ]:
x = rng.exponential(scale=1, size=10**4)

print(f"E[log X]  ≈ {np.mean(np.log(x)):.4f}   (true: −γ ≈ −0.5772)")
print(f"log E[X]  ≈ {np.log(np.mean(x)):.4f}   (true: 0)")
print(f"Jensen's inequality holds: E[log X] ≤ log E[X]? {np.mean(np.log(x)) <= np.log(np.mean(x))}")

The same idea extends to any convex or concave $g$. For $g(x) = x^3$ (convex, since $g'' = 6x > 0$ for $x > 0$) Jensen predicts $E(X^3) \geq (E X)^3$; for $g(x) = \sqrt{x}$ (concave) it predicts $E(\sqrt{X}) \leq \sqrt{E X}$.

In [ ]:
print("Convex g(x) = x³  →  E[g(X)] ≥ g(E[X]):")
print(f"  E[X³]   = {np.mean(x**3):.4f}")
print(f"  (E[X])³ = {np.mean(x)**3:.4f}")

print("\nConcave g(x) = √x  →  E[g(X)] ≤ g(E[X]):")
print(f"  E[√X]  = {np.mean(np.sqrt(x)):.4f}")
print(f"  √E[X]  = {np.sqrt(np.mean(x)):.4f}")

## Visualization of the Law of Large Numbers

The law of large numbers guarantees that the sample mean $\bar{X}_n$ converges to $\mu = E(X)$ as $n \to \infty$. A vivid way to see this is to plot the *running mean* — the cumulative average after each new observation — and watch it settle toward $\mu$.

For fair coin tosses, each $X_i \in \{0, 1\}$ with $P(X_i = 1) = 1/2$, so the running proportion of heads should converge to $1/2$. `np.cumsum` computes the running total, and dividing by the number of tosses so far gives the running mean.

In [ ]:
nsim = 300
p = 0.5
x = rng.binomial(1, p, size=nsim)
xbar = np.cumsum(x) / np.arange(1, nsim + 1)

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(x=np.arange(1, nsim + 1), y=xbar, ax=ax, label="Running proportion")
ax.axhline(p, color="black", linestyle="--", linewidth=1, label=f"p = {p}")
ax.set_ylim(0, 1)
ax.set_xlabel("Number of coin tosses")
ax.set_ylabel("Proportion of Heads")
ax.set_title("Law of Large Numbers: running proportion of Heads")
ax.legend()
plt.tight_layout()
plt.show()

The curve starts erratic and gradually homes in on $p = 0.5$. The wild swings early on are not a failure of the LLN — they are exactly what the LLN predicts: individual outcomes dominate the average when $n$ is small, and the average stabilizes only as $n$ grows.

## Monte Carlo Estimate of $\pi$

A classic application of the LLN is Monte Carlo integration. The unit disk $\{(x, y) : x^2 + y^2 \leq 1\}$ has area $\pi$ and sits inside the square $[-1, 1]^2$, which has area 4. If $(X, Y)$ is uniform on the square, then

$$P(X^2 + Y^2 \leq 1) = \frac{\pi}{4}.$$

Simulating many uniform points and recording what fraction lands inside the disk therefore estimates $\pi/4$; multiplying by 4 gives an estimate of $\pi$.

In [ ]:
nsim = 10**6
x_pts = rng.uniform(-1, 1, size=nsim)
y_pts = rng.uniform(-1, 1, size=nsim)
inside = x_pts**2 + y_pts**2 <= 1

pi_est = 4 * np.mean(inside)
print(f"Monte Carlo estimate of π: {pi_est:.5f}")
print(f"True value of π:           {np.pi:.5f}")
print(f"Error:                     {abs(pi_est - np.pi):.5f}")

With $10^6$ points the estimate is accurate to roughly 3–4 decimal places. Plotting a subset of the points makes the geometry visible: dots inside the disk are one color, dots outside another.

In [ ]:
# Use a small subset for the scatter plot
n_plot = 3000
colors = np.where(inside[:n_plot], "inside", "outside")

theta = np.linspace(0, 2 * np.pi, 500)

fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(
    x=x_pts[:n_plot], y=y_pts[:n_plot],
    hue=colors, s=4, linewidth=0, ax=ax, legend=False
)
sns.lineplot(x=np.cos(theta), y=np.sin(theta), ax=ax, color="black", linewidth=1)
ax.set_aspect("equal")
ax.set_title(rf"Monte Carlo $\pi$ estimate: {pi_est:.4f}")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

## Central Limit Theorem

The central limit theorem states that for i.i.d. random variables $X_1, X_2, \ldots$ with mean $\mu$ and finite variance $\sigma^2$, the standardized sample mean

$$\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}}$$

converges in distribution to $N(0, 1)$ as $n \to \infty$. To see this, we generate $10^4$ realizations of $\bar{X}_{12}$ — the average of 12 i.i.d. draws — and compare the histogram to the Normal approximation. We do this for two parent distributions: $\text{Unif}(0, 1)$ (symmetric, so the CLT kicks in quickly) and $\text{Expo}(1)$ (skewed, so convergence is slower).

For each distribution, we draw a $10^4 \times 12$ matrix of values and take the row means.

In [ ]:
n_rep, n_avg = 10**4, 12

xbar_unif = rng.uniform(size=(n_rep, n_avg)).mean(axis=1)
xbar_expo = rng.exponential(scale=1, size=(n_rep, n_avg)).mean(axis=1)

# Normal approximation parameters
# Unif(0,1): mean=0.5, var=1/12  →  Xbar_12 ~ N(0.5, 1/144)
# Expo(1):   mean=1,   var=1     →  Xbar_12 ~ N(1,   1/12)
unif_mu, unif_sd = 0.5, np.sqrt(1 / (12 * n_avg))
expo_mu, expo_sd = 1.0, np.sqrt(1 / n_avg)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, xbar, mu, sd, title in zip(
    axes,
    [xbar_unif, xbar_expo],
    [unif_mu,   expo_mu],
    [unif_sd,   expo_sd],
    [r"$\bar{{X}}_{{12}}$, parent Unif(0,1)",
     r"$\bar{{X}}_{{12}}$, parent Expo(1)"],
):
    x_grid = np.linspace(xbar.min(), xbar.max(), 400)
    sns.histplot(xbar, bins=60, stat="density", ax=ax, label="Simulated")
    sns.lineplot(x=x_grid, y=st.norm.pdf(x_grid, mu, sd), ax=ax,
                 label="Normal approx.")
    ax.set_xlabel(r"$\bar{x}$")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

For the Uniform parent the Normal approximation is already excellent at $n = 12$, because the Uniform is symmetric and the CLT's Berry–Esseen bound is tight. For the Exponential parent the histogram is still visibly right-skewed at $n = 12$; a larger $n$ would be needed before the Normal approximation becomes accurate.

## Chi-Square and Student-$t$ Distributions

Two distributions that arise naturally from the Normal family are the Chi-Square and the Student-$t$. If $Z_1, \ldots, Z_n$ are i.i.d. $N(0,1)$, then $Z_1^2 + \cdots + Z_n^2 \sim \chi^2_n$. The $\chi^2_n$ distribution is a special case of the Gamma: $\chi^2_n = \text{Gamma}(n/2,\, 1/2)$. `scipy.stats.chi2(df=n)` provides the PDF, CDF, and random generation.

In [ ]:
x_grid = np.linspace(0, 20, 500)

fig, ax = plt.subplots(figsize=(7, 4))
for df in [1, 2, 4, 8]:
    sns.lineplot(x=x_grid, y=st.chi2.pdf(x_grid, df=df), ax=ax, label=f"df = {df}")
ax.set_xlim(0, 20)
ax.set_ylim(0, 0.55)
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title(r"$\chi^2_n$ PDFs for several degrees of freedom")
ax.legend(title="df")
plt.tight_layout()
plt.show()

As the degrees of freedom increase, the $\chi^2_n$ distribution shifts right (its mean is $n$) and becomes more symmetric, consistent with the CLT — $\chi^2_n / n$ converges to 1 in probability.

The Student-$t_n$ distribution arises as the ratio $Z / \sqrt{V/n}$ where $Z \sim N(0,1)$ and $V \sim \chi^2_n$ are independent. It is heavier-tailed than the Normal, and the tails get lighter as $n$ increases. The $t_1$ distribution is exactly the Cauchy. `scipy.stats.t(df=n)` handles the PDF, CDF, and sampling.

In [ ]:
x_grid = np.linspace(-5, 5, 500)

fig, ax = plt.subplots(figsize=(7, 4))
for df in [1, 2, 5, 30]:
    sns.lineplot(x=x_grid, y=st.t.pdf(x_grid, df=df), ax=ax, label=f"df = {df}")
sns.lineplot(x=x_grid, y=st.norm.pdf(x_grid), ax=ax,
             label="N(0,1)", linestyle="--", color="black")
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title(r"Student-$t_n$ PDFs converging to $N(0,1)$")
ax.legend(title="df")
plt.tight_layout()
plt.show()

The $t_1$ (Cauchy) curve has extremely heavy tails; by $t_{30}$ the distribution is nearly indistinguishable from $N(0,1)$ on this scale. We can verify the $t_1 = \text{Cauchy}$ identity numerically: both PDFs should return the same value at any point.

In [ ]:
x_check = 1.5
print(f"t(df=1) PDF at x={x_check}: {st.t.pdf(x_check, df=1):.8f}")
print(f"Cauchy  PDF at x={x_check}: {st.cauchy.pdf(x_check):.8f}")